# ICHAN-DH B1: Algorithm and Controlled Verification
**Gate 5 draft — unexecuted; researcher acceptance pending.**

This notebook imports the canonical package. It implements a bounded subset of ICHAN-TV1/TV1.1, seed `20260903`, not the complete file-interface campaign. Read the operative B1 specification and Gate 5 run guide alongside it. Passing examples is implementation evidence, not a universal proof. No authentication, cover recovery, clinical safety, GPU acceleration, or measured superiority is claimed.

## I. Colab preparation
Upload a trusted source-only ZIP containing `pyproject.toml` and `src/ichan_secure/`. Do not include clinical data or credentials. Start with a fresh runtime; run cells in order. No local Python execution is required. CPU execution is sufficient; a T4 selection does not offload this code automatically.

Colab runtimes are temporary: save the executed notebook and download the evidence archive before disconnecting. [Official Colab FAQ](https://research.google.com/colaboratory/faq.html).


In [ ]:
import sys, io, json, hashlib, platform, tempfile, zipfile, stat, subprocess
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from google.colab import files

if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("This draft targets Python 3.10–3.13; review dependencies for other versions.")
if any(k == "ichan_secure" or k.startswith("ichan_secure.") for k in sys.modules):
    raise RuntimeError("Package already imported: disconnect/delete runtime and start again.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one trusted source ZIP.")
archive_name, archive_bytes = next(iter(uploaded.items()))
if not archive_name.lower().endswith(".zip"):
    raise ValueError("Expected a ZIP archive.")
WORK = Path(tempfile.mkdtemp(prefix="ichan-gate5-", dir="/content"))
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = archive.infolist()
    if len(members) > 1000 or sum(m.file_size for m in members) > 32 * 1024**2:
        raise ValueError("Archive exceeds the source-only limits.")
    seen = set()
    for member in members:
        p = PurePosixPath(member.filename)
        mode = member.external_attr >> 16
        if (p.is_absolute() or ".." in p.parts or "\\" in member.filename
                or ":" in member.filename or stat.S_ISLNK(mode)
                or member.filename in seen):
            raise ValueError("Unsafe or duplicate archive entry.")
        seen.add(member.filename)
        target = (WORK / member.filename).resolve()
        if not target.is_relative_to(WORK.resolve()):
            raise ValueError("Archive path escapes working directory.")
    archive.extractall(WORK)
roots = [p.parent for p in WORK.rglob("pyproject.toml")
         if (p.parent / "src/ichan_secure/__init__.py").is_file()]
if len(roots) != 1:
    raise ValueError("ZIP must contain exactly one package project.")
PROJECT = roots[0]
RUN_DIR = WORK / "run"
RUN_DIR.mkdir()
(RUN_DIR / "figures").mkdir()
(RUN_DIR / "source.zip").write_bytes(archive_bytes)
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(PROJECT) + "[research]", "matplotlib>=3.8,<4"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(RUN_DIR / "installation.log").write_text(install.stdout, encoding="utf-8")
print(install.stdout)
if install.returncode:
    raise RuntimeError("Installation failed; retain installation.log. Do not continue.")
# Editable installs may require path processing in an already-running notebook kernel.
import site
for directory in site.getsitepackages():
    site.addsitedir(directory)
print("Project:", PROJECT, "\nRun:", RUN_DIR)


### A. Provenance and check records
The next cell binds this run to source hashes and installed versions. Each case begins as `NOT_RUN`; failures are recorded and re-raised. Restart from a fresh runtime after changing source or dependencies. Do not erase failed runs. The pending list is an evidence obligation, not a list of optional enhancements.


In [ ]:
import importlib.metadata as metadata
import numpy as np
import matplotlib.pyplot as plt
import scipy, sklearn, pydicom, skimage, matplotlib
import ichan_secure
from ichan_secure import (
    Domain, PaddingSpec, IchanError, validate_array, classify, classify_value,
    replacement_block, guarded_depth, replace_bits, header_schedule, capacity,
    select_profile, build_header, parse_header, crc32, frame, unframe, embed, extract)
from ichan_secure.metrics import distortion_metrics, recovery_metrics, direct_map_cost

assert Path(ichan_secure.__file__).resolve().is_relative_to((PROJECT / "src").resolve())
SEED = 20260903
environment = {
    "created_utc": datetime.now(timezone.utc).isoformat(), "seed": SEED,
    "python": sys.version, "platform": platform.platform(), "execution_device": "CPU",
    "archive_sha256": hashlib.sha256(archive_bytes).hexdigest(),
    "versions": {name: metadata.version(name) for name in
                 ["numpy", "scipy", "scikit-learn", "pydicom", "scikit-image", "matplotlib"]},
    "source_sha256": {str(p.relative_to(PROJECT)): hashlib.sha256(p.read_bytes()).hexdigest()
                      for p in sorted(PROJECT.rglob("*"))
                      if p.is_file() and (p.suffix == ".py" or p.name == "pyproject.toml")
                      and "__pycache__" not in p.parts}}
for distribution, module in [("numpy",np),("scipy",scipy),("scikit-learn",sklearn),
                             ("pydicom",pydicom),("scikit-image",skimage),("matplotlib",matplotlib)]:
    if str(module.__version__) != environment["versions"][distribution]:
        raise RuntimeError("Loaded dependency differs from installed version. Restart runtime and rerun setup.")
assert environment["versions"]["pydicom"] == "3.0.2"
assert environment["versions"]["scikit-learn"] == "1.7.2"
(RUN_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
CASE_IDS = ["domain", "labels", "frame", "bootstrap", "guards", "capacity",
            "roundtrip", "malformed", "limits", "metrics", "baseline",
            "ct-roundtrip", "ct-rejections", "plots"]
PENDING = [
    "Quantized 8-to-12/12-to-16 threshold-band and physical-unit counterexample checks",
    "Complete baseline map-corruption, dependency and nonconvergence failure matrix",
    "Implicit VR Little Endian and RLE accepted file fixtures",
    "RLE BOT/fragment/EOT/size-boundary and decoder-resource failure grid",
    "One-violation-per-field metadata audit; signed/unsigned/range-padding file grid",
    "Receiver-state mutation, nested signature/MAC and content-time/extrema/loss-history grids",
    "Source/destination aliases and injected pre/post-publication/cleanup failures",
    "Independent researcher proof-by-proof review and completion of remaining TV1.1 obligations"]
record = {"suite": "ICHAN-TV1.1 bounded draft", "gate_accepted": False,
          "cases": {k: {"status": "NOT_RUN"} for k in CASE_IDS},
          "pending": PENDING, "attempts": []}
def save_record():
    (RUN_DIR / "verification_record.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
def check(identifier, function):
    if identifier not in record["cases"]:
        raise KeyError(identifier)
    entry = {"id": identifier, "started_utc": datetime.now(timezone.utc).isoformat(),
             "status": "RUNNING"}
    record["attempts"].append(entry)
    record["cases"][identifier] = entry
    save_record()
    try:
        function()
    except Exception as exc:
        entry.update(status="FAIL", error_type=type(exc).__name__, error=str(exc))
        save_record()
        raise
    entry["status"] = "PASS"
    save_record()
    print(identifier, "PASS for this fixture only")
def expect_error(code, function):
    try:
        function()
    except IchanError as exc:
        assert exc.code == code, (code, exc.code)
    else:
        raise AssertionError("Expected rejection " + code)
save_record()
print(json.dumps(environment, indent=2))


## II. Stored-domain algorithm
### A. Domain and strata — B1.1–B1.2
For stored bits $b$ and signedness $s$, $d_{min}=-s2^{b-1}$, $R=2^b-1$, and rank $z=x-d_{min}$. Compare $255z$ with $100R$ and $150R$ using integers. The inclusive boundaries partition the finite domain; monotonicity follows from multiplication by positive 255.

Pseudocode: validate domain and host dtype; reject invalid arrays; assign 0 below/equal first boundary, 1 below/equal second, otherwise 2. No rescale or windowing occurs.

**Inputs:** generated arrays only; no dataset. The checks enumerate every value for $b=2..12$ and inspect 16-bit boundaries. Signed translation and exact 8-to-16 scaling preserve the normalized rank.


In [ ]:
def domain_checks():
    for b in (2, 8, 12, 16):
        for s in (0, 1):
            d = Domain(b, s)
            assert d.minimum == -s * 2**(b-1)
            assert d.maximum - d.minimum == 2**b - 1
            a = np.array([[d.minimum, d.maximum]], dtype=np.int16 if s else np.uint16)
            assert validate_array(a, d) is a
    for bad in [np.zeros((0, 2), dtype=np.uint8), np.zeros(3, dtype=np.uint8),
                np.zeros((2, 2)), np.zeros((2, 2), dtype=bool)]:
        expect_error("E_DOMAIN", lambda a=bad: validate_array(a, Domain(8, 0)))
    expect_error("E_DOMAIN", lambda: Domain(1, 0))
    expect_error("E_DOMAIN", lambda: Domain(True, 0))
    expect_error("E_DOMAIN", lambda: validate_array(np.array([[256]], dtype=np.uint16), Domain(8, 0)))
    expect_error("E_DOMAIN", lambda: validate_array(np.zeros((1, 1), dtype=np.uint16), Domain(12, 1)))
    expect_error("E_DICOM", lambda: PaddingSpec(range_limit=1))
    expect_error("E_DICOM", lambda: PaddingSpec(2, 1))
    expect_error("E_DICOM", lambda: PaddingSpec(256).validate(Domain(8, 0)))
check("domain", domain_checks)

def label_checks():
    np.testing.assert_array_equal(classify(np.array([[99,100,101,149,150,151]], dtype=np.uint8),
                                          Domain(8,0)), [[0,0,1,1,1,2]])
    for b in range(2, 13):
        ranks = np.arange(2**b, dtype=np.int32).reshape(1,-1)
        expected = ((ranks > (100*(2**b-1))//255).astype(int)
                    + (ranks > (150*(2**b-1))//255).astype(int))
        for s in (0,1):
            d = Domain(b,s)
            got = classify((ranks+d.minimum).astype(np.int16 if s else np.uint16),d)
            np.testing.assert_array_equal(got,expected)
            assert np.all(np.diff(got.astype(int)) >= 0)
    for b in (8,12,16):
        for s in (0,1):
            d=Domain(b,s)
            for multiplier in (100,150):
                t=multiplier*d.span//255
                for z in (t-1,t,t+1):
                    expected=int(z>100*d.span//255)+int(z>150*d.span//255)
                    assert classify_value(z+d.minimum,d)==expected
    u8=np.arange(256,dtype=np.uint16).reshape(1,-1)
    np.testing.assert_array_equal(classify(u8,Domain(8,0)),
                                  classify(u8*257,Domain(16,0)))
check("labels", label_checks)


### B. Frame — B1.8; P5
The header is $4+1+8+4=17$ bytes: magic, version, big-endian payload length, and payload-only CRC-32. It consumes 136 one-bit carriers. A standalone frame appends the payload.

Pseudocode: construct fields; parse exact field widths; reject wrong magic/version/length; compare the recovered body checksum. Literal byte expectations below help avoid a shared constructor/parser error. CRC agreement is not authentication.

**Researcher check:** inspect the literal header and bit order; distinguish byte length from bit capacity.


In [ ]:
def frame_checks():
    assert crc32(b"")==0
    assert crc32(b"123456789")==0xCBF43926
    literal=bytes.fromhex("49434853010000000000000009cbf43926")
    assert build_header(b"123456789")==literal
    h=parse_header(literal)
    assert h.payload_length==9 and h.checksum==0xCBF43926
    for p in (b"", b"ICHAN-DH", bytes([0,1,127,128,254,255]), bytes(range(32))):
        assert len(frame(p))==17+len(p)
        assert unframe(frame(p))==p
    expect_error("E_LENGTH",lambda: unframe(b""))
    expect_error("E_LENGTH",lambda: unframe(frame(b"x")+b"x"))
    expect_error("E_MAGIC",lambda: parse_header(b"X"+literal[1:]))
    expect_error("E_VERSION",lambda: parse_header(literal[:4]+b"\x02"+literal[5:]))
    expect_error("E_CRC",lambda: unframe(frame(b"x")[:-1]+b"y"))
check("frame",frame_checks)


### C. Bootstrap and safe replacement — B1.3–B1.5; P1–P3
An aligned width-$r$ block starts at $d_{min}+2^r\lfloor z/2^r\rfloor$. Header eligibility requires the entire one-bit block to avoid declared padding. Select the first 136 eligible positions $J$, then reserve the entire prefix ending at $\beta=J[-1]+1$.

Body depth is the largest requested width whose entire block avoids padding and stays in one stratum. Replacement within that block preserves every profile's safe depth. Header strata need not remain unchanged.

Pseudocode: enumerate block values in a small independent oracle; require a single class and no padding; take the largest safe width. Production calls still use the canonical package. The exhaustive guard check is only for 2–4 bits, not the full 16-bit state space.


In [ ]:
def bootstrap_checks():
    d=Domain(8,0); pad=PaddingSpec(0)
    x=np.array([[0]*8+[8]*136],dtype=np.uint8)
    schedule=header_schedule(x,d,pad)
    assert schedule.positions==tuple(range(8,144)) and schedule.beta==144
    y=embed(x,b"",d,pad)
    assert header_schedule(y,d,pad)==schedule
    np.testing.assert_array_equal(y[:,:8],x[:,:8])
    assert extract(y,d,pad)==b""
    expect_error("E_BOOTSTRAP_CAPACITY",lambda: header_schedule(x[:,:143],d,pad))
    expect_error("E_CAPACITY",lambda: embed(x,b"x",d,pad))
    assert header_schedule(np.zeros((1,136),dtype=np.uint8),d).positions==tuple(range(136))
check("bootstrap",bootstrap_checks)

def oracle_class(x,d):
    z=x-d.minimum
    return int(z>100*d.span//255)+int(z>150*d.span//255)
def oracle_depth(x,d,pad,c):
    request=((1,0,0),(2,1,0),(3,2,1))[c-1][oracle_class(x,d)]
    allowed=[]
    padding=set() if pad.value is None else set(range(pad.value,(pad.range_limit if pad.range_limit is not None else pad.value)+1))
    for r in range(1,min(request,d.bits_stored)+1):
        low=d.minimum+((x-d.minimum)//(2**r))*(2**r)
        values=range(low,low+2**r)
        if all(v not in padding and oracle_class(v,d)==oracle_class(x,d) for v in values):
            allowed.append(r)
    return max([0]+allowed)
def guard_checks():
    for b in (2,3,4):
        for s in (0,1):
            d=Domain(b,s)
            for pad in (PaddingSpec(),PaddingSpec(d.minimum),PaddingSpec(d.minimum+1,d.minimum+2)):
                for x in range(d.minimum,d.maximum+1):
                    before=[guarded_depth(x,d,pad,c) for c in (1,2,3)]
                    assert before==[oracle_depth(x,d,pad,c) for c in (1,2,3)]
                    for r in before:
                        for m in range(1,r+1):
                            for v in range(2**m):
                                y=replace_bits(x,m,v,d)
                                assert d.minimum<=y<=d.maximum
                                assert (y-d.minimum)%(2**m)==v
                                assert oracle_class(y,d)==oracle_class(x,d)
                                assert [guarded_depth(y,d,pad,c) for c in (1,2,3)]==before
    assert guarded_depth(100,Domain(8,0),PaddingSpec(),3)==0
check("guards",guard_checks)


### D. Capacity and traversal — B1.6–B1.10; P4–P5
For each profile, $C_c=\sum_{i=\beta}^{N-1}a_c(x_i)$ and maximum payload bytes are $\lfloor C_c/8\rfloor$. Choose the smallest feasible profile. Raw slots of 7, 8, and 9 therefore admit 0, 1, and 1 bytes.

Pseudocode: preflight all capacities; write the fixed header; traverse the body row-major; write $m=\min(r,8L-j)$ bits; advance even at zero depth; stop at exactly $8L$. Extraction reconstructs the same state. Check literal suffix values in addition to round trips.


In [ ]:
def capacity_checks():
    d=Domain(8,0)
    for n in (7,8,9):
        x=np.array([[0]*136+[152]*n],dtype=np.uint8)
        r=capacity(x,d)
        assert r.raw_bits==(0,0,n)
        assert r.max_payload_bytes==(0,0,n//8)
        assert r.tail_bits==(0,0,n%8)
    x=np.zeros((1,144),dtype=np.uint8)
    r=capacity(x,d)
    assert r.raw_bits==(8,16,24)
    assert [select_profile(r,n) for n in (0,1,2,3)]==[1,1,2,3]
    before=x.copy()
    expect_error("E_CAPACITY",lambda: embed(x,b"1234",d))
    np.testing.assert_array_equal(x,before)
check("capacity",capacity_checks)

def roundtrip_checks():
    d=Domain(8,0)
    for length in (0,1,2,3):
        x=np.zeros((1,144),dtype=np.uint8); before=x.copy()
        p=bytes([255])*length
        y=embed(x,p,d)
        assert extract(y,d)==p
        np.testing.assert_array_equal(x,before)
        assert capacity(y,d).raw_bits==capacity(x,d).raw_bits
        assert header_schedule(y,d)==header_schedule(x,d)
        if length:
            np.testing.assert_array_equal(y[0,136:],np.full(8,2**length-1))
    x=np.zeros((1,139),dtype=np.uint8)
    y=embed(x,b"\xff",d)
    np.testing.assert_array_equal(y[0,136:],[7,7,3])
    assert extract(y,d)==b"\xff"
    # The high-stratum value has zero depth under selected profile 1.
    x=np.array([[0]*136+[255]+[0]*16],dtype=np.uint8)
    y=embed(x,b"\xff",d)
    assert y.flat[136]==255
    np.testing.assert_array_equal(y[0,145:],x[0,145:])
    assert extract(y,d)==b"\xff"
    rng=np.random.default_rng(SEED)
    for b in (8,12,16):
        for s in (0,1):
            dom=Domain(b,s)
            x=rng.integers(dom.minimum,dom.maximum+1,size=(24,24),dtype=np.int32)
            for pad in (PaddingSpec(),PaddingSpec(dom.minimum),PaddingSpec(dom.minimum,dom.minimum+3)):
                before=x.copy()
                p=bytes(range(8))
                y=embed(x,p,dom,pad)
                assert extract(y,dom,pad)==p
                np.testing.assert_array_equal(x,before)
                assert capacity(y,dom,pad).raw_bits==capacity(x,dom,pad).raw_bits
check("roundtrip",roundtrip_checks)


### E. Receiver rejection and non-reversibility — P5, P7
The receiver must reject impossible declared lengths before payload-sized allocation. The corruption helper below changes header carrier parities solely to construct fixtures; it is not a second production embedder.

Replacement is non-injective in the original cover: two different covers can yield the same output. Thus exact payload extraction is not cover recovery. Header cost is 136 placed bits, whereas reserved prefix opportunity cost is computed on the original cover and is not generally invariant.


In [ ]:
def header_fixture(array,header):
    y=array.copy()
    bits=np.unpackbits(np.frombuffer(header,dtype=np.uint8),bitorder="big")
    # This fixture uses unsigned no-padding covers, so J is literally 0..135.
    y.flat[:136]=(y.flat[:136]//2)*2+bits
    return y
def malformed_checks():
    d=Domain(8,0); x=np.zeros((1,160),dtype=np.uint8)
    h=build_header(b"")
    for code,bad in [
        ("E_MAGIC",b"X"+h[1:]),
        ("E_VERSION",h[:4]+b"\x02"+h[5:]),
        ("E_LENGTH",h[:5]+b"\xff"*8+h[13:]),
        ("E_CRC",h[:-1]+b"\x01")]:
        expect_error(code,lambda b=bad: extract(header_fixture(x,b),d))
    y=embed(x,b"\x00",d); y.flat[136]^=1
    expect_error("E_CRC",lambda: extract(y,d))
check("malformed",malformed_checks)
def limit_checks():
    d=Domain(8,0)
    a=np.zeros((1,136),dtype=np.uint8)
    b=np.ones((1,136),dtype=np.uint8)
    assert not np.array_equal(a,b)
    np.testing.assert_array_equal(embed(a,b"",d),embed(b,b"",d))
    r=capacity(a,d)
    assert r.beta==136 and r.reserved_raw_slots==(136,272,408)
    cost=direct_map_cost(5)
    assert cost.direct_symbol_bits==10 and cost.byte_packed_symbol_bytes==2
    assert cost.complete_serialized_bytes is None
check("limits",limit_checks)


## III. Controlled diagnostics
### A. Metrics and repaired baseline
For declared span $R$, $\mathrm{MSE}=N^{-1}\sum(X-Y)^2$ and $\mathrm{PSNR}=10\log_{10}(R^2/\mathrm{MSE})$; identical arrays use infinite PSNR. Cast before subtraction to avoid integer wraparound. BER is unavailable for missing or unequal-length payloads. SSIM uses the declared window, not an automatically shrunk window.

The baseline fits a fresh multinomial model and requires its external actual-width map for extraction. This small synthetic example checks that interface, not comparative effectiveness. Never compare an 8-bit baseline with a different native CT cover and call it a matched experiment.


In [ ]:
def metric_checks():
    for b in (8,12,16):
        d=Domain(b,0)
        x=np.zeros((8,8),dtype=np.uint16)
        y=np.ones((8,8),dtype=np.uint16)
        m=distortion_metrics(x,y,d)
        assert m.mse==1 and m.data_range==2**b-1
        assert np.isclose(m.psnr_db,20*np.log10(d.span))
        assert m.modified_fraction==1 and m.maximum_absolute_change==1
        identical=distortion_metrics(x,x,d)
        assert identical.mse==0 and np.isinf(identical.psnr_db)
        assert np.isclose(identical.ssim,1)
    small=distortion_metrics(np.zeros((2,2),dtype=np.uint8),np.zeros((2,2),dtype=np.uint8),Domain(8,0))
    assert small.ssim is None and small.ssim_unavailable_reason
    assert recovery_metrics(b"",b"").ber==0
    assert recovery_metrics(b"x",None).ber is None
    assert recovery_metrics(b"x",b"").ber is None
    assert recovery_metrics(b"\x00",b"\x01").ber==1/8
check("metrics",metric_checks)

def baseline_checks():
    from ichan_secure.baseline import embed_baseline,extract_baseline
    x=np.tile(np.arange(256,dtype=np.uint8),(8,1)); before=x.copy()
    p=b"ICHAN-DH"
    result=embed_baseline(x,p)
    assert extract_baseline(result.stego,result.depth_map)==p
    np.testing.assert_array_equal(x,before)
    assert sum(result.depth_map)==8*len(p)
    assert result.direct_map_bits==2*len(result.depth_map)
    expect_error("E_BASELINE_CLASSES",lambda: embed_baseline(np.zeros((16,16),dtype=np.uint8),p))
    expect_error("E_MAP",lambda: extract_baseline(x,(4,)))
    expect_error("E_LENGTH",lambda: extract_baseline(x,(1,)))
    print("Baseline observations, not expected superiority:",result.label_disagreement_count,
          result.fit_iterations,result.selected_profile)
check("baseline",baseline_checks)


### B. Restricted synthetic CT — P6
No external dataset is needed. Construct an unsigned 12-bit Explicit VR Little Endian fixture, load stored values without rescale/windowing, embed into a distinct derived file, reload, and compare pixels, receiver state, and payload. The source file must remain unchanged.

The conditional file theorem requires $\operatorname{decode}(\operatorname{save}(Y))=Y$ with unchanged domain/shape/padding state. This premise needs file evidence separate from the array proof. These fixtures target the adapter; they are **not complete, clinically valid CT IOD examples**. Other transfer syntaxes and fault-injection cases remain pending.


In [ ]:
import copy
from pydicom.dataset import FileDataset,FileMetaDataset
from pydicom.uid import CTImageStorage,ExplicitVRLittleEndian,generate_uid
from ichan_secure.dicom_io import load_dicom,embed_dicom,extract_dicom,write_derived_dicom

CT_DIR=RUN_DIR/"synthetic_dicom"
CT_DIR.mkdir(exist_ok=True)
def ct_checks():
    global CT_SOURCE,CT_DEST,CT_LOADED
    folder=Path(tempfile.mkdtemp(prefix="fixture-",dir=CT_DIR))
    CT_SOURCE,CT_DEST=folder/"source.dcm",folder/"derived.dcm"
    meta=FileMetaDataset()
    meta.MediaStorageSOPClassUID=CTImageStorage
    meta.MediaStorageSOPInstanceUID=generate_uid()
    meta.TransferSyntaxUID=ExplicitVRLittleEndian
    ds=FileDataset(str(CT_SOURCE),{},file_meta=meta,preamble=b"\0"*128)
    ds.SOPClassUID=CTImageStorage
    ds.SOPInstanceUID=meta.MediaStorageSOPInstanceUID
    ds.StudyInstanceUID=generate_uid(); ds.SeriesInstanceUID=generate_uid()
    ds.PatientName="SYNTHETIC^FIXTURE"; ds.PatientID="ICHAN-TV1"
    ds.Modality="CT"; ds.ImageType=["ORIGINAL","PRIMARY","AXIAL"]
    ds.Rows=16; ds.Columns=16; ds.SamplesPerPixel=1
    ds.PhotometricInterpretation="MONOCHROME2"
    ds.BitsAllocated=16; ds.BitsStored=12; ds.HighBit=11; ds.PixelRepresentation=0
    ds.RescaleIntercept="0"; ds.RescaleSlope="1"; ds.LossyImageCompression="00"
    ds.ContentDate="20260903"; ds.ContentTime="120000"
    ds.InstanceCreationDate="20260903"; ds.InstanceCreationTime="120000"
    pixels=(np.arange(256,dtype=np.uint16)+200).reshape(16,16)
    pixels.flat[:8]=0
    ds.add_new(0x00280120,"US",0)
    ds.add_new(0x00280106,"US",int(pixels.min()))
    ds.add_new(0x00280107,"US",int(pixels.max()))
    ds.PixelData=pixels.astype("<u2").tobytes(); ds["PixelData"].VR="OW"
    ds.save_as(CT_SOURCE,implicit_vr=False,little_endian=True,enforce_file_format=True)
    original_hash=hashlib.sha256(CT_SOURCE.read_bytes()).hexdigest()
    CT_LOADED=load_dicom(CT_SOURCE)
    np.testing.assert_array_equal(CT_LOADED.array,pixels)
    assert not CT_LOADED.array.flags.writeable
    assert CT_LOADED.domain==Domain(12,0) and CT_LOADED.padding==PaddingSpec(0)
    expected=embed(pixels,b"CT-TV1",CT_LOADED.domain,CT_LOADED.padding)
    embed_dicom(CT_SOURCE,CT_DEST,b"CT-TV1")
    loaded=load_dicom(CT_DEST); derived=loaded.dataset
    np.testing.assert_array_equal(loaded.array,expected)
    assert loaded.domain==CT_LOADED.domain and loaded.padding==CT_LOADED.padding
    assert extract_dicom(CT_DEST)==b"CT-TV1"
    assert hashlib.sha256(CT_SOURCE.read_bytes()).hexdigest()==original_hash
    assert derived.SOPInstanceUID!=ds.SOPInstanceUID
    assert derived.SeriesInstanceUID!=ds.SeriesInstanceUID
    assert derived.file_meta.MediaStorageSOPInstanceUID==derived.SOPInstanceUID
    assert list(derived.ImageType[:2])==["DERIVED","SECONDARY"]
    assert derived.SourceImageSequence[0].ReferencedSOPInstanceUID==ds.SOPInstanceUID
    assert str(derived.ContentDate)=="" and str(derived.ContentTime)==""
    assert "InstanceCreationDate" not in derived and "InstanceCreationTime" not in derived
    for key,value in [("SmallestImagePixelValue",int(expected.min())),("LargestImagePixelValue",int(expected.max()))]:
        assert int(getattr(derived,key))==value and derived.data_element(key).VR=="US"
check("ct-roundtrip",ct_checks)

def ct_rejections():
    before=hashlib.sha256(CT_DEST.read_bytes()).hexdigest()
    expect_error("E_DICOM",lambda: embed_dicom(CT_SOURCE,CT_DEST,b"x"))
    assert hashlib.sha256(CT_DEST.read_bytes()).hexdigest()==before
    bad=CT_LOADED.array.copy(); bad.flat[0]=8
    rejected=CT_SOURCE.parent/"rejected.dcm"
    expect_error("E_DICOM",lambda: write_derived_dicom(CT_SOURCE,rejected,bad))
    assert not rejected.exists()
    for field,value in [("HighBit",12),("LossyImageCompression","01")]:
        ds=copy.deepcopy(CT_LOADED.dataset); setattr(ds,field,value)
        path=CT_SOURCE.parent/("invalid-"+field+".dcm")
        ds.save_as(path,implicit_vr=False,little_endian=True,enforce_file_format=True)
        expect_error("E_DICOM",lambda p=path: load_dicom(p))
check("ct-rejections",ct_rejections)


### C. Technical visualization
Use stored-value maps and a signed change map with a symmetric scale. These are synthetic diagnostic plots; they do not imply diagnostic preservation. Histogram/ROI/cohort visualizations belong to the later analysis notebook and must match the acquired data and estimand.

**Researcher review:** compare carrier positions, the reserved prefix, and the nonzero changes. Confirm that plotting neither rescales nor modifies the array passed to the algorithm.


In [ ]:
def plot_checks():
    d=Domain(8,0)
    x=np.tile(np.arange(32,dtype=np.uint8),(16,1))
    y=embed(x,b"ICHAN-DH",d)
    delta=y.astype(np.int32)-x.astype(np.int32)
    fig,axes=plt.subplots(1,3,figsize=(12,4),constrained_layout=True)
    axes[0].imshow(x,cmap="gray",vmin=0,vmax=255); axes[0].set_title("Synthetic stored cover")
    axes[1].imshow(y,cmap="gray",vmin=0,vmax=255); axes[1].set_title("Stored stego")
    limit=max(1,int(np.abs(delta).max()))
    im=axes[2].imshow(delta,cmap="coolwarm",vmin=-limit,vmax=limit)
    axes[2].set_title("Signed stored-value change")
    fig.colorbar(im,ax=axes[2],label="Stored units")
    for ax in axes: ax.set_xlabel("Column"); ax.set_ylabel("Row")
    fig.savefig(RUN_DIR/"figures"/"stored_change.png",dpi=160)
    plt.show()
    r=capacity(x,d)
    fig,ax=plt.subplots(figsize=(6,4),constrained_layout=True)
    ax.bar([1,2,3],r.raw_bits,label="Raw suffix slots")
    ax.plot([1,2,3],r.usable_bits,"ko-",label="Byte-usable bits")
    ax.set(xticks=[1,2,3],xlabel="Profile",ylabel="Bits",title="Synthetic capacity accounting")
    ax.legend(); fig.savefig(RUN_DIR/"figures"/"capacity.png",dpi=160); plt.show()
check("plots",plot_checks)


## IV. Researcher record and export
Review `verification_record.json`: all cases must have actual outcomes, and pending obligations remain open even if the implemented subset passes. No automatic Gate 5 acceptance occurs.

If an earlier check failed after the harness initialized, you may run the export cell directly to retain that partial run; do not interpret this as permission to run dependent checks. For setup failures, retain the installation output manually. Save the executed notebook separately so its edited source and outputs are preserved.

A `DicomWriteError` with `destination_published=True` means a destination was committed before cleanup failed. Inspect it rather than retrying blindly. The fixture files contain synthetic metadata only; the package does not de-identify real data.

**Before proceeding:** record reviewer/date, expected versus observed conditions, source/environment identity, failures, and a decision for each pending obligation. Gate 6 remains unopened.


In [ ]:
import shutil
save_record()
print("Implemented case statuses:")
print(json.dumps(record["cases"],indent=2))
print("Pending obligations:",*PENDING,sep="\n- ")
print("Gate accepted:",record["gate_accepted"])
# Excludes the ZIP itself by placing the archive beside RUN_DIR.
export_path=shutil.make_archive(str(WORK/"gate5-evidence"),"zip",root_dir=RUN_DIR)
files.download(export_path)
